In [20]:
import time
import gspread
from riotwatcher import LolWatcher, ApiError
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

In [21]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")

In [22]:
# Configuración de Clientes
scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive",
]
creds = ServiceAccountCredentials.from_json_keyfile_name("credentials.json", scope)
client = gspread.authorize(creds)

# 1. Extraer el ID desde la URL del documento en el navegador.
# Formato: https://docs.google.com/spreadsheets/d/<ID_DEL_DOCUMENTO>/edit
spreadsheet_id = "1_mDRZEKVEVdwC96vS_Kkktn2aGP7I8LUZynwxXLP31s"

spreadsheet = client.open_by_key(spreadsheet_id)

# 2. Utilizar acceso directo por llave
sheet = spreadsheet.worksheet("Camilo Arias")

In [23]:
def insertar_dataframe_mapeado(sheet, df, mapping, start_row=None):
    """
    Inserta un DataFrame en Google Sheets usando un mapeo de columnas.

    Args:
        sheet: Objeto worksheet de gspread.
        df: DataFrame de pandas con los datos.
        mapping: Diccionario {Columna_Sheet: Columna_Pandas}
                 Ej: {'D': 'my_champ', 'F': 'win_loss', 'H': 'rival'}
        start_row: Fila inicial. Si es None, busca la siguiente vacía en 'D'.
    """
    if start_row is None:
        # Buscamos la primera fila vacía basándonos en la columna D
        start_row = len(sheet.col_values(4)) + 1

    payload = []

    # Convertimos DF a lista de diccionarios para iteración eficiente
    records = df.to_dict("records")

    for i, row_data in enumerate(records):
        current_row = start_row + i

        for sheet_col, df_col in mapping.items():
            if df_col in row_data:
                val = row_data[df_col]

                # Agregamos cada celda al lote de actualización
                payload.append(
                    {"range": f"{sheet_col}{current_row}", "values": [[val]]}
                )

    # Ejecución en una sola transacción para optimizar cuota de API
    if payload:
        sheet.batch_update(payload, value_input_option="USER_ENTERED")
        print(f"✅ Se insertaron {len(records)} filas exitosamente.")


In [24]:
watcher = LolWatcher(API_KEY)
region_route = "americas"
platform_route = "la1"

In [25]:
from riotwatcher import RiotWatcher, LolWatcher, ApiError

api_key = API_KEY
riot_watcher = RiotWatcher(api_key)
lol_watcher = LolWatcher(api_key)


def obtener_puuid(riot_client, game_name, tag_line, continental_route="americas"):
    try:
        # Petición obligatoria a través de la instancia de RiotWatcher
        account_data = riot_client.account.by_riot_id(
            region=continental_route, game_name=game_name, tag_line=tag_line
        )
        return account_data["puuid"]

    except ApiError as err:
        if err.response.status_code == 404:
            print("Error: Riot ID no encontrado. Verifique la ortografía y el tagLine.")
        elif err.response.status_code == 403:
            print("Error: Clave de API inválida o expirada.")
        else:
            print(f"Error de API: {err}")
        return None


# Ejecución correcta
my_puuid = obtener_puuid(riot_watcher, "Camilo Arias", "LAN")

In [26]:
my_puuid

'P21ORAvQ93csHTyh01SkGB27-Ocf13wUDZD0OBix9H-c5yr5BVOxDajVQH6LJp7GZnvjf_RBS8RJgw'

In [27]:
from datetime import datetime
import pytz


def formatear_fecha(timestamp_ms: int) -> str:
    # Convertir ms a segundos
    fecha_utc = datetime.fromtimestamp(timestamp_ms / 1000, tz=pytz.utc)
    # Localizar a Bogotá
    zona_bogota = pytz.timezone("America/Bogota")
    fecha_local = fecha_utc.astimezone(zona_bogota)
    return fecha_local.strftime("%Y-%m-%d %H:%M:%S")

In [28]:
def obtener_participant_id(json_partida, mi_puuid):
    """
    Realiza un mapeo cruzado en el JSON de la partida para localizar
    el participantId dinámico (1-10) asociado al PUUID estático.
    """
    try:
        # Se asume que json_partida puede ser el Match DTO o el Timeline DTO
        participantes = json_partida["info"]["participants"]

        participant_id = next(
            p["participantId"] for p in participantes if p["puuid"] == mi_puuid
        )
        return participant_id

    except StopIteration:
        # Failsafe si el bloque de datos no corresponde al jugador
        raise ValueError(
            f"Fallo de integridad: El PUUID {mi_puuid} no existe en este registro."
        )
    except KeyError as e:
        raise KeyError(f"Error estructural en el JSON. Llave no encontrada: {e}")

In [29]:
def obtener_partidas_pendientes(sheet, puuid, region="americas"):
    registrados = set(sheet.col_values(3))

    try:
        ultima_fecha_str = sheet.col_values(4)[-1]
        ultima_fecha_dt = datetime.strptime(ultima_fecha_str, "%Y-%m-%d %H:%M:%S")
        # Convertir a epoch segundos (Riot usa segundos para startTime, no ms)
        start_time_epoch = int(ultima_fecha_dt.timestamp())
    except (IndexError, ValueError):
        start_time_epoch = int(datetime(2026, 1, 1, tzinfo=pytz.utc).timestamp())
        print(
            f"Hoja vacía o dato inválido. Iniciando ingesta desde 2026-01-01 (epoch: {start_time_epoch})"
        )

    # 3. Consultar Riot API con filtro de tiempo
    nuevos_ids = lol_watcher.match.matchlist_by_puuid(
        region=region,
        puuid=puuid,
        start_time=start_time_epoch,
        count=100,  # Aumentar rango ya que el filtrado por tiempo es eficiente
        type="ranked",
    )

    # 4. Deduplicación final en memoria
    pendientes = [m_id for m_id in nuevos_ids if m_id not in registrados]

    return pendientes

In [30]:
def process_toplane_matches(match_list, puuid, watcher, region_route="americas"):
    datos_partidas = []

    for match_id in match_list:
        try:
            # 1. Consulta inicial del resumen de la partida
            match = watcher.match.by_id(region_route, match_id)

            if match["info"].get("gameDuration", 0) < 600:
                print(f"Partida {match_id} omitida: Duracion Insuficiente")
                continue

            # 2. Identificación del participante y validación del rol
            me_data = next(
                p for p in match["info"]["participants"] if p["puuid"] == puuid
            )

            # Salida temprana si no es Toplane
            if me_data["teamPosition"] != "TOP":
                continue

            # 3. Consulta del timeline (Solo se ejecuta si el rol es TOP)
            timeline = watcher.match.timeline_by_match(region_route, match_id)
            me_id = me_data["participantId"]

            # 4. Identificación del Rival (Toplaner enemigo)
            try:
                rival_data = next(
                    p
                    for p in match["info"]["participants"]
                    if p["teamPosition"] == "TOP" and p["puuid"] != puuid
                )
                rival_id = rival_data["participantId"]
            except StopIteration:
                # Manejo de casos atípicos (AFK desde el inicio, roles mal asignados por la API)
                rival_data = {"championName": "Desconocido/AFK"}
                rival_id = None

            def get_frame_stats(frame_idx, p_id):
                # Validación de longitud del timeline (Surrenders tempranos)
                if frame_idx >= len(timeline["info"]["frames"]):
                    return {"cs": 0, "gold": 0, "xp": 0}

                frame = timeline["info"]["frames"][frame_idx]["participantFrames"][
                    str(p_id)
                ]
                return {
                    "cs": frame["minionsKilled"] + frame["jungleMinionsKilled"],
                    "gold": frame["totalGold"],
                    "xp": frame["xp"],
                }

            # Extracción de mis métricas
            f8_me = get_frame_stats(8, me_id)
            f14_me = get_frame_stats(14, me_id)

            # Extracción de métricas del rival (Si existe)
            if rival_id:
                f8_rival = get_frame_stats(8, rival_id)
                f14_rival = get_frame_stats(14, rival_id)

            kda_str = f"{me_data['kills']}/{me_data['deaths']}/{me_data['assists']}"
            game_length_min = (
                max(
                    1,
                    me_data["challenges"].get(
                        "gameLength", me_data.get("gameDuration", 1)
                    ),
                )
                / 60
            )

            row = [
                match_id,
                formatear_fecha(match["info"]["gameStartTimestamp"]),
                "Solo/Duo" if match["info"]["queueId"] == 420 else "Flex",
                me_data["championName"],
                rival_data["championName"],
                "W" if me_data["win"] else "L",
                kda_str,
                round(
                    (me_data["totalMinionsKilled"] + me_data["neutralMinionsKilled"])
                    / game_length_min,
                    2,
                ),
                f8_me["cs"],
                f14_me["cs"],
                f8_rival["cs"] if rival_id else "N/A",
                f14_rival["cs"] if rival_id else "N/A",
                f8_me["gold"],
                f14_me["gold"],
                f8_rival["gold"] if rival_id else "N/A",
                f14_rival["gold"] if rival_id else "N/A",
                f8_me["xp"],
                f14_me["xp"],
                f8_rival["xp"] if rival_id else "N/A",
                f14_rival["xp"] if rival_id else "N/A",
                me_data["damageDealtToTurrets"],
            ]
            datos_partidas.append(row)

            # Control de Rate Limit básico (Requerido para procesos secuenciales)
            time.sleep(1.5)

        except ApiError as err:
            if err.response.status_code == 429:
                print(
                    f"Rate limit excedido en la partida {match_id}. Implementar backoff."
                )
                break
            else:
                print(f"Error en la partida {match_id}: {err}")
                continue

    return datos_partidas


matches = obtener_partidas_pendientes(sheet, my_puuid, region_route)

toplane_data = process_toplane_matches(
    matches,
    my_puuid,
    watcher,
)

toplane_data

[['LA1_1716402734',
  '2026-05-07 18:11:55',
  'Solo/Duo',
  'Jax',
  'Nasus',
  'W',
  '13/5/4',
  5.37,
  42,
  88,
  49,
  92,
  2139,
  4465,
  3043,
  5118,
  3162,
  7394,
  3720,
  7539,
  6825],
 ['LA1_1716392571',
  '2026-05-07 17:28:09',
  'Solo/Duo',
  'Camille',
  'Briar',
  'W',
  '10/11/4',
  5.77,
  51,
  95,
  56,
  101,
  2313,
  4704,
  3480,
  6461,
  3573,
  6701,
  4698,
  7924,
  16718],
 ['LA1_1716386349',
  '2026-05-07 16:50:22',
  'Solo/Duo',
  'Rumble',
  'Kayle',
  'W',
  '8/5/13',
  5.8,
  52,
  103,
  43,
  79,
  3168,
  5314,
  2151,
  3801,
  4131,
  8788,
  3139,
  7077,
  2526]]

In [31]:
schema_toplane_data = {
    "match_id": "string",
    "fecha_partida": "string",
    "queue_type": "string",
    "my_champion": "string",
    "rival_champion": "string",
    "result": "string",
    "kda": "string",
    "cspm": "float",
    "f8_cs": "int",
    "f14_cs": "int",
    "f8_rival_cs": "int",
    "f14_rival_cs": "int",
    "f8_gold": "int",
    "f14_gold": "int",
    "f8_rival_gold": "int",
    "f14_rival_gold": "int",
    "f8_xp": "int",
    "f14_xp": "int",
    "f8_rival_xp": "int",
    "f14_rival_xp": "int",
    "damage_to_turrets": "int",
}

In [32]:
df_toplane = pd.DataFrame(toplane_data, columns=schema_toplane_data.keys())

In [33]:
df_toplane

,match_id,fecha_partida,queue_type,my_champion,rival_champion,result,kda,cspm,f8_cs,f14_cs,...,f14_rival_cs,f8_gold,f14_gold,f8_rival_gold,f14_rival_gold,f8_xp,f14_xp,f8_rival_xp,f14_rival_xp,damage_to_turrets
0,LA1_1716402734,2026-05-07 18:11:55,Solo/Duo,Jax,Nasus,W,13/5/4,5.37,42,88,...,92,2139,4465,3043,5118,3162,7394,3720,7539,6825
1,LA1_1716392571,2026-05-07 17:28:09,Solo/Duo,Camille,Briar,W,10/11/4,5.77,51,95,...,101,2313,4704,3480,6461,3573,6701,4698,7924,16718
2,LA1_1716386349,2026-05-07 16:50:22,Solo/Duo,Rumble,Kayle,W,8/5/13,5.80,52,103,...,79,3168,5314,2151,3801,4131,8788,3139,7077,2526


In [34]:
df_toplane.columns

Index(['match_id', 'fecha_partida', 'queue_type', 'my_champion',
       'rival_champion', 'result', 'kda', 'cspm', 'f8_cs', 'f14_cs',
       'f8_rival_cs', 'f14_rival_cs', 'f8_gold', 'f14_gold', 'f8_rival_gold',
       'f14_rival_gold', 'f8_xp', 'f14_xp', 'f8_rival_xp', 'f14_rival_xp',
       'damage_to_turrets'],
      dtype='str')

In [35]:
cols_numericas = [
    "f8_cs",
    "f8_rival_cs",
    "f14_cs",
    "f14_rival_cs",
    "f8_gold",
    "f8_rival_gold",
    "f14_gold",
    "f14_rival_gold",
    "f8_xp",
    "f8_rival_xp",
    "f14_xp",
    "f14_rival_xp",
]

In [36]:
for col in cols_numericas:
    df_toplane[col] = pd.to_numeric(df_toplane[col], errors="coerce")

In [37]:
df_toplane = df_toplane.assign(
    f8_cs_diff=lambda df_: df_["f8_cs"] - df_["f8_rival_cs"],
    f14_cs_diff=lambda df_: df_["f14_cs"] - df_["f14_rival_cs"],
    f8_gold_diff=lambda df_: df_["f8_gold"] - df_["f8_rival_gold"],
    f14_gold_diff=lambda df_: df_["f14_gold"] - df_["f14_rival_gold"],
    f8_xp_diff=lambda df_: df_["f8_xp"] - df_["f8_rival_xp"],
    f14_xp_diff=lambda df_: df_["f14_xp"] - df_["f14_rival_xp"],
    kda="'"
    + df_toplane["kda"].astype(
        str
    ),  # evitar que google sheet lo identifique como fecha
).sort_values(by="fecha_partida", ascending=True)

In [38]:
config_mapeo = {
    "C": "match_id",
    "D": "fecha_partida",
    "E": "queue_type",
    "G": "my_champion",
    "J": "rival_champion",
    "M": "result",
    "N": "kda",
    "O": "cspm",
    "P": "f8_cs_diff",
    "Q": "f14_cs_diff",
    "R": "f8_gold_diff",
    "S": "f14_gold_diff",
    "T": "f8_xp_diff",
    "U": "f14_xp_diff",
    "V": "damage_to_turrets",
}

insertar_dataframe_mapeado(sheet, df_toplane, config_mapeo)

✅ Se insertaron 3 filas exitosamente.
